In [1]:
import torch
import triton
import triton.language as tl 

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Triton:", triton.__version__)



DEVICE = triton.runtime.driver.active.get_active_torch_device()

PyTorch: 2.14.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Triton: 3.8.0


In [2]:
@triton.jit
def add_kernal(x_ptr,
               y_ptr,
               output_ptr,
               num_elements,
               Block_size:tl.constexpr
               ):
    prid = tl.program_id(axis=0)
    block_thred = prid*Block_size
    offset = block_thred + tl.arange(0,Block_size)

    mask = offset<num_elements

    x = tl.load(x_ptr+offset,mask=mask)
    y = tl.load(y_ptr+offset,mask=mask)

    output = x+y

    tl.store(output_ptr+offset,output,mask=mask)



def add(x:torch.Tensor,
        y: torch.Tensor,
        ):
    output = torch.empty_like(x)

    assert x.device == DEVICE and  y.device == DEVICE and output.device == DEVICE
    num_element = x.numel()
    grid = lambda meta: (triton.cdiv(num_element, meta['Block_size']), )

    add_kernal[grid](x,y,output_ptr=output,num_elements=num_element,Block_size=1024)

    return output

    

    

In [3]:
torch.manual_seed(0)
size = 98432
x = torch.rand(size, device=DEVICE)
y = torch.rand(size, device=DEVICE)
output_torch = x + y
output_triton = add(x, y)
print(output_torch)
print(output_triton)
print(f'The maximum difference between torch and triton is '
      f'{torch.max(torch.abs(output_torch - output_triton))}')

tensor([1.3713, 1.3076, 0.4940,  ..., 0.9592, 0.3409, 1.2567], device='cuda:0')
tensor([1.3713, 1.3076, 0.4940,  ..., 0.9592, 0.3409, 1.2567], device='cuda:0')
The maximum difference between torch and triton is 0.0
